In [1]:
r = range(10000, 1000000000)
r[45006230]

45016230

In [2]:
class Letters(object):
    def __init__(self):
        self.current = 'a'
    def __next__(self):
        if self.current > 'd':
            raise StopIteration
        result = self.current
        self.current = chr(ord(result)+1)
        return result
    def __iter__(self):
        return self
    
letters = Letters()

In [3]:
letters.__next__()

'a'

In [4]:
letters.__next__()

'b'

In [5]:
letters.__next__()

'c'

In [6]:
letters.__next__()

'd'

In [7]:
letters.__next__()

StopIteration: 

In [9]:
letters = Letters()
for letter in letters:
    print(letter)

a
b
c
d


In [10]:
counts = [1, 2, 3]
i = counts.__iter__()
try:
    while True:
        item = i.__next__()
        print(item)
except StopIteration:
    pass

1
2
3


In [11]:
def letters_generator():
    current = 'a'
    while current <= 'd':
        yield current
        current = chr(ord(current)+1)
        
for letter in letters_generator():
    print(letter)

a
b
c
d


In [13]:
letters = letters_generator()
type(letters)

generator

In [14]:
letters.__next__()

'a'

In [15]:
letters.__next__()

'b'

In [16]:
letters.__next__()

'c'

In [17]:
letters.__next__()

'd'

In [18]:
letters.__next__()

StopIteration: 

In [20]:
def all_pairs(s):
    for item1 in s:
        for item2 in s:
            yield (item1, item2)

list(all_pairs([1, 2, 3]))

[(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]

In [21]:
class LetterIterable(object):
    def __iter__(self):
        current = 'a'
        while current <= 'd':
            yield current
            current = chr(ord(current)+1)

letters = LetterIterable()
all_pairs(letters).__next__()

('a', 'a')

In [22]:
class Stream(object):
    """A lazily computed recursive list."""
    def __init__(self, first, compute_rest, empty=False):
        self.first = first
        self._compute_rest = compute_rest
        self.empty = empty
        self._rest = None
        self._computed = False
    @property
    def rest(self):
        """Return the rest of the stream, computing it if necessary."""
        assert not self.empty, 'Empty streams have no rest.'
        if not self._computed:
            self._rest = self._compute_rest()
            self._computed = True
        return self._rest
    def __repr__(self):
        if self.empty:
            return '<empty stream>'
        return 'Stream({0}, <compute_rest>)'.format(repr(self.first))
    
Stream.empty = Stream(None, None, True)

In [23]:
s = Stream(1, lambda: Stream(2+3, lambda: Stream.empty))

In [24]:
s.first

1

In [25]:
s.rest.first

5

In [32]:
def make_integer_stream(first=1):
    def compute_rest():
        return make_integer_stream(first+1)
    return Stream(first, compute_rest)

def map_stream(fn, s):
    if s.empty:
        return s
    def compute_rest():
        return map_stream(fn, s.rest)
    return Stream(fn(s.first), compute_rest)

def filter_stream(fn, s):
    if s.empty:
        return s
    def compute_rest():
        return filter_stream(fn, s.rest)
    if fn(s.first):
        return Stream(s.first, compute_rest)
    return compute_rest()

def truncate_stream(s, k):
    if s.empty or k == 0:
        return Stream.empty
    def compute_rest():
        return truncate_stream(s.rest, k-1)
    return Stream(s.first, compute_rest)

def stream_to_list(s):
    r = []
    while not s.empty:
        r.append(s.first)
        s = s.rest
    return r

In [33]:
s = make_integer_stream(3)

In [34]:
s.first

3

In [35]:
s.rest

Stream(4, <compute_rest>)

In [36]:
s.rest.first

4

In [37]:
s.rest.rest

Stream(5, <compute_rest>)

In [39]:
m = map_stream(lambda x: x*x, s)
stream_to_list(truncate_stream(m, 5))

[9, 16, 25, 36, 49]

In [40]:
def primes(pos_stream):
    def not_divible(x):
        return x % pos_stream.first != 0
    def compute_rest():
        return primes(filter_stream(not_divible, pos_stream.rest))
    return Stream(pos_stream.first, compute_rest)

In [41]:
p1 = primes(make_integer_stream(2))
stream_to_list(truncate_stream(p1, 7))

[2, 3, 5, 7, 11, 13, 17]

In [42]:
def match(pattern):
    print('Looking for ' + pattern)
    try:
        while True:
            s = (yield)
            if pattern in s:
                print(s)
    except GeneratorExit:
        print("=== Done ===")

In [43]:
m = match("Jabberwock")

In [44]:
m.__next__()

Looking for Jabberwock


In [45]:
type(m)

generator

In [46]:
m.send("the Jabberwock with eyes of flame")

the Jabberwock with eyes of flame


In [47]:
m.send("came whiffling through the tulgey wood")

In [48]:
m.close()

=== Done ===


In [49]:
text = 'Commending spending is offending to people pending lending!'
matcher = match('ending')
matcher.__next__()

Looking for ending


In [50]:
read(text, matcher)

NameError: name 'read' is not defined

In [52]:
def match_filter(pattern, next_coroutine):
    print('Looking for ' + pattern)
    try:
        while True:
            s = (yield)
            if pattern in s:
                next_coroutine.send(s)
    except GeneratorExit:
        next_coroutine.close()
        
def print_consumer():
    print('Preparing to print')
    try:
        while True:
            line = (yield)
            print(line)
    except GeneratorExit:
        print("=== Done ===")

In [53]:
printer = print_consumer()
printer.__next__()

Preparing to print


In [54]:
matcher = match_filter('pend', printer)
matcher.__next__()

=== Done ===
Looking for pend


In [55]:
read(text, matcher)

NameError: name 'read' is not defined

In [56]:
def count_letters(next_coroutine):
    try:
        while True:
            s = (yield)
            counts = {letter:s.count(letter) for letter in set(s)}
            next_coroutine.send(counts)
    except GeneratorExit as e:
        next_coroutine.close()
        
def sum_dictionaries():
    total = {}
    try:
        while True:
            counts = (yield)
            for letter, count in counts.items():
                total[letter] = count + total.get(letter, 0)
    except GeneratorExit:
        max_letter = max(total.items(), key=lambda t: t[1])[0]
        print("Most frequent letter: " + max_letter)

In [58]:
s = sum_dictionaries()

In [59]:
s.__next__()

In [60]:
c = count_letters(s)

In [61]:
c.__next__()

In [62]:
read(text, c)

NameError: name 'read' is not defined

In [63]:
def read_to_many(text, coroutines):
    for word in text.split():
        for coroutine in coroutines:
            coroutine.send(word)
    for coroutine in coroutines:
        coroutine.close()

In [64]:
m = match("mend")

In [65]:
m.__next__()

Looking for mend


In [66]:
p = match("pe")

In [67]:
p.__next__()

Looking for pe


In [68]:
read_to_many(text, [m, p])

Commending
spending
people
pending
=== Done ===
=== Done ===
